# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 dataset package using the `mlcroissant` library. All dataset elements, including record sets, fields, and columns, are referenced by their `@id` to maintain interoperability.

### Dataset Source
The dataset is defined by a Croissant schema and available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and records using the `mlcroissant.Dataset` API. The metadata provides detailed information on the dataset, including its description, authors, and record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore the available record sets and their fields. We'll display `@id`s for each to ensure precise references across all analysis steps.

In [ ]:
# List all record sets with their @id
record_sets = dataset.record_sets
print("Available record sets (referenced by @id):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    fields = rs.get('field', [])
    print(f"  Fields:")
    for fld in fields:
        print(f"    - Field @id: {fld['@id']} (name: {fld.get('name', '(no name)')})")
    print()

## 3. Data Extraction

Extract tables for each record set by their `@id`. We'll create a DataFrame for each, using the record set and field `@id`s identified above. All columns will be referenced using their Croissant IDs.

In [ ]:
# Get a list of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Extract records for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  First rows:")
        print(df.head())
        print()
    else:
        print(f"No records found for RecordSet @id: {rs_id}\n")

## 4. Exploratory Data Analysis (EDA)

Here we demonstrate processing steps using Croissant `@id` references for fields. We'll filter, normalize, and group data from one of the record sets. Please refer to the overview above for valid numeric and grouping field `@id`s for your dataset.

In [ ]:
# Example: Select a record set to explore
# (Choose a RecordSet @id from the overview, e.g. the first one)
if len(dataframes) > 0:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    print(f"Numeric fields in RecordSet '{rs_id}': {numeric_fields}")

    # Select the first numeric field for demonstration
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]  # This is the @id of the field/column
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by another field (e.g. second column)
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if len(group_fields) > 0:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric fields found in this RecordSet.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Visualize distributions or relationships using the DataFrames extracted by `@id`. For example, plot the numeric field distribution or a grouped average.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and len(numeric_fields) > 0:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet {rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, bar plot group means
    if 'grouped_df' in locals():
        grouped_df.plot(kind='bar', figsize=(8,4), legend=False)
        plt.title(f"Mean of '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()


## 6. Conclusion

This notebook has guided you through loading, exploring, and processing the FAIR^2 clinical/genetic dataset using the `mlcroissant` library. All operations referenced dataset entities via their Croissant `@id`, ensuring clarity and reproducibility. You can adapt the notebook for deeper analysis or alternate record sets, always referencing fields and columns via `@id`.